# EnerGIS - District Heating Optimization

Komplettes Notebook für Optimierung, Analyse und Visualisierung.

## Inhalt
1. **Setup** - Imports und Konfiguration
2. **Optimierung** - Workflow ausführen
3. **Ergebnisse** - KPIs und Zusammenfassung
4. **Netzwerk** - Thermische Netzwerk-Analyse
5. **Visualisierung** - Plots und Dashboard

---

## 1. Setup

In [ ]:
# Bootstrap
import sys
from pathlib import Path

current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break

from energis.io.notebook_helpers import setup_notebook_environment
PROJECT_ROOT = setup_notebook_environment()

print(f"\n✅ Projekt: {PROJECT_ROOT}")

In [ ]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime

from energis.run import rolling_horizon as rh
from energis.config.merge import load_and_merge
from energis.io.notebook_helpers import (
    save_workflow_run,
    display_workflow_summary,
    display_kpi_summary,
    list_saved_workflows,
    load_workflow_from_saved
)

print("✅ Imports erfolgreich")

## 2. Konfiguration

Passe hier die Einstellungen an:

**⚠️ AKTUELLER STATUS:**
- `enforce: true` - Terminal Constraints AKTIVIERT (für Storage Cycling)
- `fix_design: true` - Design nach 1. Window fixiert (verhindert Investment-Konflikte)
- `thermal_network.enabled: false` - Temporär DEAKTIVIERT (zum Testen)

In [ ]:
# ============================================================
# KONFIGURATION - Hier anpassen!
# ============================================================

# Option 1: Einzelne Config-Datei (empfohlen)
CONFIG_PATHS = ['configs/stadtbach.yaml']

# Option 2: Mehrere Config-Dateien (für Varianten)
# CONFIG_PATHS = [
#     'configs/base.yaml',
#     'configs/tech_catalog.yaml',
#     'configs/systems/baseline.yaml',
#     'configs/scenarios/full_year.yaml',
# ]

# Optional: Parameter überschreiben
OVERRIDES = None
# OVERRIDES = {
#     'scenario': {'horizon': {'end': '2023-01-31 23:00'}},
#     'costs': {'co2_price_eur_per_t': 150.0}
# }

# ============================================================

# Prüfen
print("📋 Konfiguration:")
for cfg in CONFIG_PATHS:
    exists = (PROJECT_ROOT / cfg).exists()
    print(f"  {'✅' if exists else '❌'} {cfg}")

# Vorschau
cfg = load_and_merge(CONFIG_PATHS)
print(f"\n🔧 Einstellungen:")
print(f"  Solver:     {cfg.get('run', {}).get('solver', 'N/A')}")
print(f"  CO2-Preis:  {cfg.get('costs', {}).get('co2_price_eur_per_t', 'N/A')} EUR/t")
print(f"  Netzwerk:   {'Ja' if cfg.get('thermal_network', {}).get('enabled', False) else 'Nein'}")

## 3. Optimierung ausführen

In [ ]:
%%time
print("=" * 70)
print("🚀 STARTE OPTIMIERUNG")
print("=" * 70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "=" * 70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("=" * 70)
    print(f"Workflow: {' → '.join(workflow.plan.steps)}")
    
    SUCCESS = True
    
except Exception as e:
    print(f"\n❌ FEHLER: {e}")
    import traceback
    traceback.print_exc()
    workflow = None
    SUCCESS = False

## 4. Ergebnisse speichern

In [ ]:
if SUCCESS and workflow:
    workflow_dir = save_workflow_run(
        workflow,
        name="EnerGIS Simulation",
        description="Optimierungslauf",
        config_paths=CONFIG_PATHS
    )
    print(f"\n📂 Gespeichert in: {workflow_dir}")
else:
    print("⚠️ Keine Ergebnisse zum Speichern")

## 5. Zusammenfassung & KPIs

In [ ]:
if SUCCESS and workflow:
    display_workflow_summary(workflow)
    print("\n")
    display_kpi_summary(workflow)
else:
    print("⚠️ Keine Ergebnisse verfügbar")

## 6. Thermisches Netzwerk

In [ ]:
if SUCCESS and workflow:
    network_enabled = workflow.config.get('thermal_network', {}).get('enabled', False)
    
    if network_enabled:
        print("🌡️ THERMISCHES NETZWERK")
        print("=" * 70)
        
        result = workflow.pf_result or workflow.rh_result
        
        if result and hasattr(result, 'summary') and 'thermal_network' in result.summary:
            net = result.summary['thermal_network']
            
            print(f"\n📏 Topologie:")
            print(f"  Knoten:        {net.get('Number_of_nodes', 0)}")
            print(f"  Rohre:         {net.get('Number_of_pipes', 0)}")
            print(f"  Länge:         {net.get('Total_pipe_length_m', 0):.0f} m")
            
            print(f"\n🔥 Wärme:")
            print(f"  Geliefert:     {net.get('Total_heat_delivered_MWh', 0):.1f} MWh")
            print(f"  Verluste:      {net.get('Total_heat_loss_MWh', 0):.1f} MWh")
            print(f"  Verlustrate:   {net.get('Heat_loss_percentage', 0):.2f}%")
        else:
            print("\n⚠️ Netzwerk-Ergebnisse nicht verfügbar")
    else:
        print("ℹ️ Thermisches Netzwerk nicht aktiviert")
        print("   Aktivieren in configs/stadtbach.yaml:")
        print("   thermal_network:")
        print("     enabled: true")

## 7. Visualisierung

In [ ]:
if SUCCESS and workflow:
    from energis.io.publication_plotter import export_publication_plots
    
    result = workflow.pf_result or workflow.rh_result
    
    if result:
        print("📊 Erstelle Plots...")
        
        plots = export_publication_plots(
            outdir=str(workflow_dir) if 'workflow_dir' in dir() else 'exports',
            table=result.table,
            series=result.series,
            summary_sections=result.summary if hasattr(result, 'summary') else {},
            dpi=150,
            formats=("png",),
            plot_types=["heat_balance", "electric_balance", "storage"]
        )
        
        print(f"✅ {len(plots)} Plots erstellt")
else:
    print("⚠️ Keine Daten für Visualisierung")

## 8. Interaktive Zeitreihen

In [ ]:
if SUCCESS and workflow:
    result = workflow.pf_result or workflow.rh_result
    
    if result:
        # DataFrame erstellen
        ts = pd.DataFrame({
            'timestamp': result.table.index,
            **{col: result.table.data[col] for col in result.table.columns},
            **result.series,
        })
        ts.set_index('timestamp', inplace=True)
        
        print(f"📋 Zeitreihen: {len(ts)} Zeilen, {len(ts.columns)} Spalten")
        print(f"\nVerfügbare Spalten:")
        for i, col in enumerate(ts.columns[:20]):
            print(f"  {col}")
        if len(ts.columns) > 20:
            print(f"  ... und {len(ts.columns) - 20} weitere")
        
        display(ts.head())

In [ ]:
# Einfacher Plot mit matplotlib
if SUCCESS and workflow and 'ts' in dir():
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Wärmebedarf
    if 'heatd' in ts.columns:
        axes[0].plot(ts.index, ts['heatd'], label='Wärmebedarf', color='red', alpha=0.7)
        axes[0].set_ylabel('Leistung [MW]')
        axes[0].legend()
        axes[0].set_title('Wärmebedarf')
    
    # Strompreis
    if 'price' in ts.columns:
        axes[1].plot(ts.index, ts['price'], label='Strompreis', color='blue', alpha=0.7)
        axes[1].set_ylabel('Preis [EUR/MWh]')
        axes[1].legend()
        axes[1].set_title('Strompreis')
    
    plt.tight_layout()
    plt.show()

## 9. Dashboard starten

Für detaillierte interaktive Analyse:

In [ ]:
print("🎛️ Dashboard starten:")
print("\n  python start_dashboard.py")
print("\nOder direkt hier (nur in Jupyter):")

In [ ]:
# Dashboard im Notebook anzeigen (optional)
# Auskommentieren um zu aktivieren:

# if SUCCESS and workflow:
#     from energis.io.dashboard import create_dashboard
#     dashboard = create_dashboard(workflow)
#     dashboard.servable()

## 10. Gespeicherte Workflows laden

In [ ]:
# Liste aller gespeicherten Workflows
saved = list_saved_workflows(sort_by="date")

print(f"📦 {len(saved)} gespeicherte Workflows:\n")
for i, wf in enumerate(saved[:5], 1):
    name = wf['name'][:50]
    print(f"  {i}. {name}")

In [ ]:
# Workflow laden (Index anpassen)
# LOAD_INDEX = 0  # Erster Workflow
# 
# if saved:
#     loaded_workflow = load_workflow_from_saved(saved[LOAD_INDEX]['path'])
#     print(f"✅ Geladen: {saved[LOAD_INDEX]['name']}")
#     display_workflow_summary(loaded_workflow)

---

## Hilfe

**Dokumentation:**
- `README.md` - Übersicht
- `docs/methodology.md` - Methodik

**CLI-Nutzung:**
```bash
python -m energis.run configs/stadtbach.yaml
```

**Dashboard als Webapp:**
```bash
python start_dashboard.py
```